# 05 — Region Classification

So far we can answer `True` or `False`: is a point inside a polygon?

That's geometry. What we want is *meaning*:

```text
"This point is in Sector Alpha."
"Sector Alpha covers the Northern Gulf and Iran."
"Tehran is the reference target for this sector."
```

The geometry tells us *which feature*. The feature's **properties** tell us *what that means*. This notebook builds the layer between raw coordinates and human-readable classification.

## Setup

In [ ]:
import json
from pathlib import Path
from ipyleaflet import Map, GeoJSON, basemaps
from ipywidgets import Output

DATA_DIR = Path("data")

with open(DATA_DIR / "regions.geojson") as f:
    regions_fc = json.load(f)


def point_in_ring(lon, lat, ring):
    n = len(ring) - 1
    inside = False
    for i in range(n):
        xi, yi = ring[i][0],   ring[i][1]
        xj, yj = ring[i+1][0], ring[i+1][1]
        if (yi > lat) != (yj > lat):
            x_intersect = xi + (lat - yi) * (xj - xi) / (yj - yi)
            if lon < x_intersect:
                inside = not inside
    return inside

def point_in_feature(lon, lat, feature):
    geom = feature["geometry"]
    if geom["type"] != "Polygon":
        return False
    return point_in_ring(lon, lat, geom["coordinates"][0])

print(f"Loaded {len(regions_fc['features'])} regions.")

## Extracting Properties

Every GeoJSON Feature has a `properties` object — a dictionary of arbitrary key-value pairs. The geometry says *where*; the properties say *what*.

In [ ]:
# Inspect properties of all regions
print("Region properties:")
print()
for feat in regions_fc["features"]:
    props = feat["properties"]
    print(f"  name:        {props.get('name', '—')}")
    print(f"  description: {props.get('description', '—')}")
    print()

## The `classify_point` Function

This is the core of the module. Given a point, it returns a structured classification result — not just which polygon, but the full properties dict plus a flag for "no match".

In [ ]:
def classify_point(lon, lat, feature_collection):
    """
    Find which region contains (lon, lat) and return a classification result.

    Returns a dict:
      {
        'found': True/False,
        'lon': lon,
        'lat': lat,
        'name': region name or None,
        'properties': full properties dict or {},
        'feature': the matched feature or None
      }
    """
    for feature in feature_collection["features"]:
        if point_in_feature(lon, lat, feature):
            return {
                "found": True,
                "lon": lon,
                "lat": lat,
                "name": feature["properties"].get("name"),
                "properties": feature["properties"],
                "feature": feature
            }
    return {
        "found": False,
        "lon": lon,
        "lat": lat,
        "name": None,
        "properties": {},
        "feature": None
    }


# Test classification
cities = [
    ("Tehran",   51.388,  35.695),
    ("Riyadh",   46.675,  24.688),
    ("Cairo",    31.235,  30.044),
    ("Madrid",   -3.703,  40.417),
    ("Muscat",   58.593,  23.614),
]

print(f"{'City':<12}  {'Found':>6}  {'Sector':<22}  {'Description'}")
print("─" * 70)
for city_name, lon, lat in cities:
    result = classify_point(lon, lat, regions_fc)
    sector = result["name"] or "—"
    desc   = result["properties"].get("description", "—")
    print(f"  {city_name:<10}  {str(result['found']):>6}  {sector:<22}  {desc}")

## Handling the "No Match" Case

When a point is outside all regions, the function returns `found=False`. Callers must handle this explicitly — otherwise any code that accesses `result['name']` will get `None` and crash downstream.

Two patterns for handling no-match:

In [ ]:
# Pattern 1: check the 'found' flag
def describe_location(lon, lat):
    result = classify_point(lon, lat, regions_fc)
    if result["found"]:
        return f"Point ({lon:.2f}, {lat:.2f}) is in {result['name']} — {result['properties'].get('description', '')}"
    else:
        return f"Point ({lon:.2f}, {lat:.2f}) is outside all known sectors."

# Pattern 2: use .get() with a default
def get_sector_name(lon, lat, default="Unknown"):
    result = classify_point(lon, lat, regions_fc)
    return result["name"] or default


test_cases = [
    (51.388, 35.695),   # Tehran — inside
    (-3.703, 40.417),   # Madrid — outside
]

for lon, lat in test_cases:
    print(describe_location(lon, lat))
    print(f"  Sector name (with default): {get_sector_name(lon, lat)}")
    print()

## Building a Classification Map

We can colorize a set of test points by their classification result — inside points get the sector color, outside points get gray.

In [ ]:
sector_colors = {
    "Sector Alpha":   "#e74c3c",
    "Sector Bravo":   "#e67e22",
    "Sector Charlie": "#2980b9",
    "Sector Delta":   "#27ae60",
    "Sector Echo":    "#8e44ad",
    None: "#95a5a6"   # gray for no match
}

test_points = [
    {"name": "Tehran",   "lon": 51.388,  "lat": 35.695},
    {"name": "Riyadh",   "lon": 46.675,  "lat": 24.688},
    {"name": "Cairo",    "lon": 31.235,  "lat": 30.044},
    {"name": "Madrid",   "lon": -3.703,  "lat": 40.417},
    {"name": "Muscat",   "lon": 58.593,  "lat": 23.614},
    {"name": "Baghdad",  "lon": 44.361,  "lat": 33.338},
    {"name": "Aden",     "lon": 45.036,  "lat": 12.779},
]

m = Map(center=(28, 45), zoom=3, basemap=basemaps.CartoDB.Positron)

# Add sector polygons
for feat, color in zip(regions_fc["features"], list(sector_colors.values())[:5]):
    m.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        style={"color": color, "fillColor": color, "fillOpacity": 0.12, "weight": 2}
    ))

# Add classified test points — colored by their sector
for pt in test_points:
    result = classify_point(pt["lon"], pt["lat"], regions_fc)
    color  = sector_colors.get(result["name"], sector_colors[None])
    feat = {
        "type": "Feature",
        "properties": {"name": pt["name"], "sector": result["name"] or "none"},
        "geometry": {"type": "Point", "coordinates": [pt["lon"], pt["lat"]]}
    }
    m.add(GeoJSON(
        data={"type": "FeatureCollection", "features": [feat]},
        style={"color": color, "fillColor": color, "fillOpacity": 1.0, "weight": 2}
    ))

print("Points colored by their classified sector (gray = no match):")
for pt in test_points:
    r = classify_point(pt["lon"], pt["lat"], regions_fc)
    print(f"  {pt['name']:<10}  →  {r['name'] or '(no match)'}")
m

## Exercise A

Connect `classify_point` to a click handler.

1. Create a map showing all five sectors.
2. When the user clicks anywhere on the map, print the classification result:
   - If inside a sector: print the sector name and description.
   - If outside all sectors: print `"Outside all sectors"`.
3. Include the click coordinates (lat, lon) in the output.

In [ ]:
from ipyleaflet import Map, GeoJSON, basemaps
from ipywidgets import Output
import json
from pathlib import Path

DATA_DIR = Path("data")
with open(DATA_DIR / "regions.geojson") as f:
    regions_fc = json.load(f)

out = Output()

# Map + click handler
# On click: extract lat/lon, flip to lon/lat, classify_point, print result
# Your code here

## Exercise B

Handle the "outside all sectors" case gracefully in a pipeline.

Write a function `safe_classify(lon, lat)` that:
1. Returns a dict with `name`, `description`, and `status` keys.
2. If inside a sector: `status = "classified"`, `name` and `description` filled from properties.
3. If outside: `status = "unclassified"`, `name = "Unknown"`, `description = "Outside operational sectors"`.
4. Test it on at least 3 cities inside sectors and 2 outside.

In [ ]:
def safe_classify(lon, lat):
    # Returns {name, description, status}
    # status: "classified" or "unclassified"
    pass  # your code here

test_locs = [
    ("Tehran",   51.388,  35.695),
    ("Riyadh",   46.675,  24.688),
    ("Baghdad",  44.361,  33.338),
    ("Madrid",   -3.703,  40.417),
    ("Honolulu", -157.855, 21.305),
]

for name, lon, lat in test_locs:
    result = safe_classify(lon, lat)
    print(f"{name}: status={result['status']}, sector={result['name']}")

---

## Check Your Understanding

**1.** Why is it better to return a structured result dict from `classify_point` rather than just returning the region name as a string?

**2.** What should a well-designed function return when a point is outside all polygons — `None`, an empty string, a special object, or something else? What are the tradeoffs?

```python
# No code needed — answer in your own words
```

## Next

In [06 — Interactive Click Applications](./06_Interactive_Click_Applications.ipynb), we assemble the full pipeline — click, flip, classify, display — and connect it to the blast zone buffers from module 10 for a complete interactive spatial query system.